In [1]:
import importlib
import math
import os
import random
import time
import json
from pathlib import Path

import hydra
import numpy as np
import pandas as pd
import torch
from omegaconf import DictConfig, OmegaConf
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

from src.utils import (
    get_func_build_dataset,
    build_model,
    build_optimizer,
    build_scheduler,
    build_criterion,
    train_one_epoch,
    validate,
    set_seed,
    save_results,
)

import wandb


In [2]:


def run(cfg):
    print(cfg)

    run_name = (
        f"{cfg.dataset.name}_{cfg.model.name}_{cfg.epoch_window.name}_sub-{cfg.subject}"
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    with wandb.init(
            project=cfg.wandb.project,
            mode=cfg.wandb.mode,
            name=run_name,
            config=OmegaConf.to_container(cfg, resolve=True),
    ):

        func_build_dataset = get_func_build_dataset(cfg.model.func_build_dataset)
        train_loader, valid_loader, test_loader = func_build_dataset(cfg)

        model = build_model(cfg)
        model.to(device)

        for pipeline_name in cfg.model.pipeline:

            if cfg.model[pipeline_name].trainable != "all":
                for param in model.parameters():
                    param.requires_grad = False

                for layer_name in cfg.model[pipeline_name].trainable:
                    layer = model.get_submodule(layer_name)
                    for param in layer.parameters():
                        param.requires_grad = True

            for name, param in model.named_parameters():
                print(name, param.requires_grad)

            optimizer = build_optimizer(model, cfg.model[pipeline_name].optimizer)
            scheduler = build_scheduler(optimizer, cfg.model[pipeline_name].scheduler)
            criterion = build_criterion(cfg.model[pipeline_name].criterion)

            best_acc = 0
            save_name = f"{cfg.model.batch_size}_epochs_{cfg.model[pipeline_name].n_epochs}_batch_size_{cfg.model[pipeline_name].optimizer.kwargs.lr}_lr"
            best_model_path = (
                    Path("model")
                    / cfg.model.name
                    / f"{cfg.model.name}_{cfg.dataset.name}_{cfg.epoch_window.name}_{cfg.subject}_{save_name}.pt"
            )
            best_model_path.parent.mkdir(exist_ok=True, parents=True)

            for epoch in range(cfg.model[pipeline_name].n_epochs):
                t_start = time.time()
                train_loss, train_acc = train_one_epoch(
                    model,
                    train_loader,
                    optimizer,
                    criterion,
                    device=device,
                )
                valid_loss, valid_acc = validate(
                    model,
                    valid_loader,
                    criterion,
                    device=device,
                )
                duration = time.time() - t_start

                if cfg.model[pipeline_name].scheduler.need_loss:
                    scheduler.step(valid_loss)
                else:
                    scheduler.step()

                if valid_acc > best_acc:
                    best_acc = valid_acc
                    torch.save(model.state_dict(), best_model_path)

                print(
                    f"{epoch + 1:03d}, train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, valid_loss: {valid_loss:.4f}, valid_acc: {valid_acc:.4f}, duration: {duration:.2f}s, lr: {scheduler.get_last_lr()[0]:.4f}")
                wandb.log(
                    {
                        "train_loss": train_loss,
                        "train_accuracy": train_acc,
                        "valid_loss": valid_loss,
                        "valid_accuracy": valid_acc,
                        "best_valid_accuracy": best_acc,
                        "time_one_epoch": duration,
                        "lr": scheduler.get_last_lr()[0],
                    }
                )

            model.load_state_dict(torch.load(best_model_path))
        test_loss, test_acc = validate(model, test_loader, criterion, device=device)
        wandb.log({"test_loss": test_loss, "test_accuracy": test_acc})

        save_results(cfg, test_acc)
        print(
            f"Done: {cfg.model.name} | {cfg.dataset.name} | {cfg.epoch_window.name} | subject {cfg.subject} → {test_acc:.4f}"
        )


from hydra import initialize, compose


def main(cfg):
    set_seed(cfg.seed)
    run(cfg)


with initialize(version_base=None, config_path="conf"):
    cfg = compose(config_name="config",
                  overrides=["model=REVE"])

#print(cfg)
#print(cfg.model)

main(cfg)


{'dataset': {'name': 'Dreyer2023', 'subjects': {'n_subjects': 87, 'exclude': [59]}, 'path': './Dataset/Dreyer2023', 'n_sessions': 1, 'runs': {'train': [1, 2], 'test': [3, 4, 5, 6]}, 'epoch_windows': [[0, 4], [0.5, 4.5]], 'n_channels': 27, 'n_classes': 2, 'input_window_samples': {'none': 2048, 200: 800}}, 'model': {'name': 'REVE', 'sfreq': 200, 'batch_size': 256, 'training_step': 2, 'pipeline': ['linear_probing', 'fine_tuning'], 'func_build_dataset': 'src.utils.build_dataset_REVE', 'linear_probing': {'n_epochs': 100, 'trainable': ['final_layer'], 'optimizer': {'module': 'torch.optim', 'name': 'AdamW', 'kwargs': {'lr': 0.001, 'weight_decay': 0.0005}}, 'scheduler': {'module': 'torch.optim.lr_scheduler', 'name': 'CosineAnnealingLR', 'kwargs': {'T_max': 99}, 'need_loss': False}, 'criterion': {'module': 'torch.nn', 'name': 'CrossEntropyLoss', 'kwargs': None}}, 'fine_tuning': {'n_epochs': 5, 'trainable': 'all', 'optimizer': {'module': 'torch.optim', 'name': 'AdamW', 'kwargs': {'lr': 0.001, 'w

Traceback (most recent call last):
  File "/tmp/ipykernel_1008361/1133780564.py", line 51, in run
    train_loss, train_acc = train_one_epoch(
                            ^^^^^^^^^^^^^^^^
  File "/home/skojima/git/MI_bci_epoch_window_influence/src/utils.py", line 405, in train_one_epoch
KeyboardInterrupt


KeyboardInterrupt: 